In [ ]:
from google.colab import files
uploaded=files.upload()

Saving ml_anomaly_results.csv to ml_anomaly_results (2).csv
Saving high_risk_cases.csv to high_risk_cases (1).csv
Saving transactions.csv to transactions (1).csv
Saving settlements.csv to settlements (2).csv
Saving refunds.csv to refunds (2).csv
Saving fees.csv to fees (2).csv


In [ ]:
import pandas as pd
import numpy as np

transactions = pd.read_csv("transactions.csv")
fees = pd.read_csv("fees.csv")
refunds = pd.read_csv("refunds.csv")
settlements = pd.read_csv("settlements.csv")

ml_results = pd.read_csv("ml_anomaly_results.csv")
high_risk = pd.read_csv("high_risk_cases.csv")

print("Transactions:", len(transactions))
print("Fees:", len(fees))
print("Refunds:", len(refunds))
print("Settlements:", len(settlements))
print("ML results:", len(ml_results))
print("High-risk cases:", len(high_risk))

Transactions: 2000
Fees: 2000
Refunds: 152
Settlements: 1947
ML results: 2000
High-risk cases: 33


In [ ]:
print("Transaction columns:")
print(transactions.columns.tolist())

print("\nML columns:")
print(ml_results.columns.tolist())

Transaction columns:
['transaction_id', 'order_id', 'customer_id', 'amount', 'payment_date', 'payment_status']

ML columns:
['transaction_id', 'order_id', 'customer_id', 'amount', 'expected_settlement', 'settled_amount', 'difference', 'status', 'priority', 'anomaly_score', 'ml_risk', 'combined_risk_score', 'risk_level']


In [ ]:
def get_transaction(transaction_id):

    result = transactions[
        transactions["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        return {
            "error": f"Transaction {transaction_id} not found."
        }

    return result.iloc[0].to_dict()

In [ ]:
sample_id = transactions.iloc[0]["transaction_id"]

print(
    get_transaction(sample_id)
)

{'transaction_id': 'TXN000001', 'order_id': 'ORD000001', 'customer_id': 'CUST00655', 'amount': 2872.14, 'payment_date': '2026-08-24', 'payment_status': 'SUCCESS'}


In [ ]:
def get_fee_details(transaction_id):

    result = fees[
        fees["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        return {
            "error": f"No fee record found for {transaction_id}."
        }

    return result.iloc[0].to_dict()

In [ ]:
print(
    get_fee_details(sample_id)
)

{'transaction_id': 'TXN000001', 'gateway_fee': 32.58, 'tax_on_fee': 5.86}


In [ ]:
def get_refund_details(transaction_id):

    result = refunds[
        refunds["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        return {
            "transaction_id": transaction_id,
            "refund_count": 0,
            "total_refund": 0
        }

    return {
        "transaction_id": transaction_id,
        "refund_count": len(result),
        "total_refund": round(
            result["refund_amount"].sum(),
            2
        ),
        "refund_records": result.to_dict(
            orient="records"
        )
    }

In [ ]:
print(
    get_refund_details(sample_id)
)

{'transaction_id': 'TXN000001', 'refund_count': 0, 'total_refund': 0}


In [ ]:
def get_settlement_details(transaction_id):

    result = settlements[
        settlements["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        return {
            "error": f"No settlement found for {transaction_id}."
        }

    return result.iloc[0].to_dict()

In [ ]:
print(
    get_settlement_details(sample_id)
)

{'settlement_id': 'SET000001', 'transaction_id': 'TXN000001', 'settled_amount': 2833.7, 'settlement_date': '2026-08-26'}


In [ ]:
def get_ml_risk(transaction_id):

    result = ml_results[
        ml_results["transaction_id"].astype(str)
        == str(transaction_id)
    ]

    if result.empty:
        return {
            "error": f"No ML result found for {transaction_id}."
        }

    row = result.iloc[0]

    return {
        "transaction_id": transaction_id,
        "anomaly_score": float(
            row["anomaly_score"]
        ),
        "ml_risk": row["ml_risk"],
        "combined_risk_score": float(
            row["combined_risk_score"]
        ),
        "risk_level": row["risk_level"]
    }

In [ ]:
print(
    get_ml_risk(sample_id)
)

{'transaction_id': 'TXN000001', 'anomaly_score': 15.63, 'ml_risk': 'LOW', 'combined_risk_score': 9.38, 'risk_level': 'LOW'}


In [ ]:
def investigate_transaction(transaction_id):

    transaction = get_transaction(
        transaction_id
    )

    fee = get_fee_details(
        transaction_id
    )

    refund = get_refund_details(
        transaction_id
    )

    settlement = get_settlement_details(
        transaction_id
    )

    ml = get_ml_risk(
        transaction_id
    )

    return {
        "transaction": transaction,
        "fee": fee,
        "refund": refund,
        "settlement": settlement,
        "ml_risk": ml
    }

In [ ]:
case = investigate_transaction(
    sample_id
)

case

{'transaction': {'transaction_id': 'TXN000001',
  'order_id': 'ORD000001',
  'customer_id': 'CUST00655',
  'amount': 2872.14,
  'payment_date': '2026-08-24',
  'payment_status': 'SUCCESS'},
 'fee': {'transaction_id': 'TXN000001',
  'gateway_fee': 32.58,
  'tax_on_fee': 5.86},
 'refund': {'transaction_id': 'TXN000001',
  'refund_count': 0,
  'total_refund': 0},
 'settlement': {'settlement_id': 'SET000001',
  'transaction_id': 'TXN000001',
  'settled_amount': 2833.7,
  'settlement_date': '2026-08-26'},
 'ml_risk': {'transaction_id': 'TXN000001',
  'anomaly_score': 15.63,
  'ml_risk': 'LOW',
  'combined_risk_score': 9.38,
  'risk_level': 'LOW'}}

In [ ]:
SYSTEM_PROMPT = """
You are LedgerLens AI Investigator,
an AI assistant for financial reconciliation.

Your job is to investigate suspicious or
high-risk payment transactions using ONLY
the evidence provided by LedgerLens.

Rules:

1. Never invent transaction information.
2. Never invent financial amounts.
3. Clearly distinguish facts from hypotheses.
4. Explain the financial discrepancy.
5. Consider the ML anomaly score.
6. Give a practical recommendation.
7. If evidence is insufficient, say so.
8. Do not authorize or execute financial transactions.
9. Recommendations must be sent for human review.
10. Be concise and professional.

Return the investigation using:

Finding:
Evidence:
Financial Impact:
Possible Cause:
Recommendation:
Confidence:
"""

In [ ]:
test_transaction = high_risk.iloc[0]["transaction_id"]

print("Testing transaction:", test_transaction)

evidence = investigate_transaction(test_transaction)

print(evidence)

Testing transaction: TXN001741
{'transaction': {'transaction_id': 'TXN001741', 'order_id': 'ORD001741', 'customer_id': 'CUST00394', 'amount': 23274.06, 'payment_date': '2026-08-13', 'payment_status': 'SUCCESS'}, 'fee': {'transaction_id': 'TXN001741', 'gateway_fee': 550.36, 'tax_on_fee': 99.06}, 'refund': {'transaction_id': 'TXN001741', 'refund_count': 0, 'total_refund': 0}, 'settlement': {'error': 'No settlement found for TXN001741.'}, 'ml_risk': {'transaction_id': 'TXN001741', 'anomaly_score': 99.81, 'ml_risk': 'CRITICAL', 'combined_risk_score': 99.08, 'risk_level': 'CRITICAL'}}


In [ ]:
investigate_transaction(test_transaction)

{'transaction': {'transaction_id': 'TXN001741',
  'order_id': 'ORD001741',
  'customer_id': 'CUST00394',
  'amount': 23274.06,
  'payment_date': '2026-08-13',
  'payment_status': 'SUCCESS'},
 'fee': {'transaction_id': 'TXN001741',
  'gateway_fee': 550.36,
  'tax_on_fee': 99.06},
 'refund': {'transaction_id': 'TXN001741',
  'refund_count': 0,
  'total_refund': 0},
 'settlement': {'error': 'No settlement found for TXN001741.'},
 'ml_risk': {'transaction_id': 'TXN001741',
  'anomaly_score': 99.81,
  'ml_risk': 'CRITICAL',
  'combined_risk_score': 99.08,
  'risk_level': 'CRITICAL'}}

In [ ]:
import json

def prepare_evidence(transaction_id):
    evidence = investigate_transaction(transaction_id)

    return json.dumps(
        evidence,
        indent=2,
        default=str
    )

print("prepare_evidence function created successfully.")

prepare_evidence function created successfully.


In [37]:
test_transaction = high_risk.iloc[0]["transaction_id"]

print("Testing transaction:", test_transaction)

evidence = prepare_evidence(test_transaction)

print(evidence)

Testing transaction: TXN001741
{
  "transaction": {
    "transaction_id": "TXN001741",
    "order_id": "ORD001741",
    "customer_id": "CUST00394",
    "amount": 23274.06,
    "payment_date": "2026-08-13",
    "payment_status": "SUCCESS"
  },
  "fee": {
    "transaction_id": "TXN001741",
    "gateway_fee": 550.36,
    "tax_on_fee": 99.06
  },
  "refund": {
    "transaction_id": "TXN001741",
    "refund_count": 0,
    "total_refund": 0
  },
  "settlement": {
    "error": "No settlement found for TXN001741."
  },
  "ml_risk": {
    "transaction_id": "TXN001741",
    "anomaly_score": 99.81,
    "ml_risk": "CRITICAL",
    "combined_risk_score": 99.08,
    "risk_level": "CRITICAL"
  }
}


In [38]:
# ============================================
# LedgerLens - AI Investigation (Day 4)
# ============================================

def ai_investigate(transaction_id):
    evidence = prepare_evidence(transaction_id)

    prompt = f"""
You are LedgerLens, an AI financial investigation assistant.

Analyze ONLY the financial evidence provided below.

Rules:
1. Identify suspicious or abnormal financial activity.
2. Explain the evidence clearly.
3. Calculate or mention financial impact when supported by the evidence.
4. Do not invent missing information.
5. If evidence is insufficient, explicitly say so.
6. Do not authorize or execute financial transactions.
7. Recommendations are for human review only.
8. Be concise and professional.

Return the investigation using exactly:

Finding:
Evidence:
Financial Impact:
Possible Cause:
Recommendation:
Confidence:

FINANCIAL EVIDENCE:
{evidence}
"""

    print(prompt)

In [39]:
ai_investigate(test_transaction)


You are LedgerLens, an AI financial investigation assistant.

Analyze ONLY the financial evidence provided below.

Rules:
1. Identify suspicious or abnormal financial activity.
2. Explain the evidence clearly.
3. Calculate or mention financial impact when supported by the evidence.
4. Do not invent missing information.
5. If evidence is insufficient, explicitly say so.
6. Do not authorize or execute financial transactions.
7. Recommendations are for human review only.
8. Be concise and professional.

Return the investigation using exactly:

Finding:
Evidence:
Financial Impact:
Possible Cause:
Recommendation:
Confidence:

FINANCIAL EVIDENCE:
{
  "transaction": {
    "transaction_id": "TXN001741",
    "order_id": "ORD001741",
    "customer_id": "CUST00394",
    "amount": 23274.06,
    "payment_date": "2026-08-13",
    "payment_status": "SUCCESS"
  },
  "fee": {
    "transaction_id": "TXN001741",
    "gateway_fee": 550.36,
    "tax_on_fee": 99.06
  },
  "refund": {
    "transaction_id": 

In [40]:
# ============================================
# LedgerLens - Gemini AI Connection
# ============================================

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [42]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hello to LedgerLens in one sentence."
)

print(response.text)

Hello, LedgerLens, it is a pleasure to connect with a platform focused on bringing clarity, precision, and insight to financial data.


In [43]:
def ai_investigate(evidence):

    prompt = f"""
You are LedgerLens, an AI financial investigation assistant.

Analyze ONLY the financial evidence provided below.

Rules:
1. Identify suspicious or abnormal financial activity.
2. Explain the evidence clearly.
3. Calculate or mention financial impact when supported by the evidence.
4. Do not invent missing information.
5. If evidence is insufficient, explicitly say so.
6. Do not authorize or execute financial transactions.
7. Recommendations are for human review only.
8. Be concise and professional.

Return exactly:

Finding:
Evidence:
Financial Impact:
Possible Cause:
Recommendation:
Confidence:

FINANCIAL EVIDENCE:
{evidence}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [44]:
result = ai_investigate(evidence)

print(result)

Finding:
A high-value successful transaction (TXN001741) has been flagged with a critical ML risk level and is missing its corresponding settlement record.

Evidence:
- **Transaction Amount**: $23,274.06 (Status: SUCCESS on 2026-08-13).
- **Fees**: Gateway fee of $550.36 and tax on fee of $99.06 are recorded.
- **Settlement**: "No settlement found for TXN001741."
- **ML Risk Metrics**: Anomaly score of 99.81 and combined risk score of 99.08, both categorized as "CRITICAL".

Financial Impact:
- **Unsettled Funds**: $23,274.06 remains outstanding and has not been cleared or received.
- **Incurred Fees**: Direct cost of $649.42 ($550.36 gateway fee + $99.06 tax on fee) has been charged despite the lack of settlement.

Possible Cause:
- The transaction may have been blocked or held by the payment processor due to high fraud risk, or a technical integration failure occurred between the payment gateway and the settlement system.

Recommendation:
1. Halt fulfillment of order ORD001741 immedia

In [45]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 6.9 MB/s eta 0:00:00


In [46]:
from pypdf import PdfReader

In [47]:
from pypdf import PdfReader

pdf_path = "/content/LedgerLens_Test_Financial_Data.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print(text)

LedgerLens – Test Financial Evidence
Sample payment-gateway report for testing PDF upload, financial evidence extraction, reconciliation, and AI
investigation in the LedgerLens project.
Transaction Details
 Field
Value
Transaction ID
TXN001741
Order ID
ORD001741
Customer ID
CUST00394
Amount
$23,274.06
Payment Date
2026-08-13
Payment Status
SUCCESS
Fee Details
 Fee Type
Amount
Gateway Fee
$550.36
Tax on Fee
$99.06
Total Fees
$649.42
Refund Details
 Refund Count
0
Total Refund
$0.00
Settlement Details
 Field
Value
Settlement Status
NOT FOUND
Settlement ID
N/A
Settlement Date
N/A
Gateway Message
No settlement found for TXN001741.
Risk Information
 Metric
Value
Anomaly Score
99.81
Combined Risk Score
99.08
Risk Level
CRITICAL

Investigation Test Prompt
Investigate this transaction using only the evidence in this document. Identify suspicious activity, explain the
evidence, calculate the financial impact where supported, identify a possible cause, provide a recommendation for
human review, 

In [48]:
def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [49]:
pdf_text = extract_pdf_text(pdf_path)

print(pdf_text)

LedgerLens – Test Financial Evidence
Sample payment-gateway report for testing PDF upload, financial evidence extraction, reconciliation, and AI
investigation in the LedgerLens project.
Transaction Details
 Field
Value
Transaction ID
TXN001741
Order ID
ORD001741
Customer ID
CUST00394
Amount
$23,274.06
Payment Date
2026-08-13
Payment Status
SUCCESS
Fee Details
 Fee Type
Amount
Gateway Fee
$550.36
Tax on Fee
$99.06
Total Fees
$649.42
Refund Details
 Refund Count
0
Total Refund
$0.00
Settlement Details
 Field
Value
Settlement Status
NOT FOUND
Settlement ID
N/A
Settlement Date
N/A
Gateway Message
No settlement found for TXN001741.
Risk Information
 Metric
Value
Anomaly Score
99.81
Combined Risk Score
99.08
Risk Level
CRITICAL

Investigation Test Prompt
Investigate this transaction using only the evidence in this document. Identify suspicious activity, explain the
evidence, calculate the financial impact where supported, identify a possible cause, provide a recommendation for
human review, 

In [50]:
def prepare_pdf_evidence(pdf_text):
    evidence = {
        "source": "PDF",
        "content": pdf_text
    }

    return evidence

In [51]:
pdf_evidence = prepare_pdf_evidence(pdf_text)

print(pdf_evidence)

{'source': 'PDF', 'content': 'LedgerLens – Test Financial Evidence\nSample payment-gateway report for testing PDF upload, financial evidence extraction, reconciliation, and AI\ninvestigation in the LedgerLens project.\nTransaction Details\n Field\nValue\nTransaction ID\nTXN001741\nOrder ID\nORD001741\nCustomer ID\nCUST00394\nAmount\n$23,274.06\nPayment Date\n2026-08-13\nPayment Status\nSUCCESS\nFee Details\n Fee Type\nAmount\nGateway Fee\n$550.36\nTax on Fee\n$99.06\nTotal Fees\n$649.42\nRefund Details\n Refund Count\n0\nTotal Refund\n$0.00\nSettlement Details\n Field\nValue\nSettlement Status\nNOT FOUND\nSettlement ID\nN/A\nSettlement Date\nN/A\nGateway Message\nNo settlement found for TXN001741.\nRisk Information\n Metric\nValue\nAnomaly Score\n99.81\nCombined Risk Score\n99.08\nRisk Level\nCRITICAL\n\nInvestigation Test Prompt\nInvestigate this transaction using only the evidence in this document. Identify suspicious activity, explain the\nevidence, calculate the financial impact wh

In [52]:
pdf_result = ai_investigate(pdf_evidence)

print(pdf_result)

Finding:
A successful transaction of high value ($23,274.06) has failed to settle and has been flagged with a critical risk status and extremely high anomaly scores.

Evidence:
*   **Transaction ID:** TXN001741
*   **Amount:** $23,274.06 (Payment Status: SUCCESS)
*   **Total Fees:** $649.42 (Gateway Fee: $550.36, Tax: $99.06)
*   **Settlement Status:** NOT FOUND (Gateway Message: "No settlement found for TXN001741.")
*   **Risk Metrics:** Anomaly Score: 99.81, Combined Risk Score: 99.08, Risk Level: CRITICAL

Financial Impact:
The unsettled amount is $23,274.06. Total fees calculated/charged for this transaction are $649.42. The total outstanding or withheld funds equal $23,274.06.

Possible Cause:
The settlement may have been withheld or blocked by the gateway due to the transaction's critical risk flags (Anomaly Score of 99.81 and Combined Risk Score of 99.08), or a technical integration error prevented settlement generation.

Recommendation:
For human review:
1. Contact the payment 